#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF & Krea2Edit integration tools.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!pip install -q uv
!cd {LOCAL_WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121
!uv pip install --system -q gguf torchsde websocket-client

KREA_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/comfyui-krea2edit")
if not os.path.exists(KREA_NODE_DIR):
    print("🎨 [@CoinNoin] Installing Krea2Edit Processing Nodes...")
    !git clone https://github.com/lbouaraba/comfyui-krea2edit {KREA_NODE_DIR} &> /dev/null
else:
    !cd {KREA_NODE_DIR} && git pull &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 [@CoinNoin] Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !uv pip install --system -q -r {GGUF_NODE_DIR}/requirements.txt
else:
    !cd {GGUF_NODE_DIR} && git pull &> /dev/null

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader & LoRA Setup
#@markdown The **Krea-2 Turbo Q6_K GGUF** and required base assets are fetched automatically. You can optionally download or upload extra LoRAs.

# --- Optional LoRA Source ---
LORA_SOURCE = "None (Use Default Krea2 Identity LoRA)" #@param ["None (Use Default Krea2 Identity LoRA)", "Download from URL", "Upload from Computer"]
LORA_URLS = "" #@param {type:"string"}

import os
import subprocess
import urllib.parse
from google.colab import files

WORKSPACE = "/content/ComfyUI"

# Fixed Models for Krea2Edit
FIXED_UNET_URL = "https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF/resolve/main/krea2_turbo-Q6_K.gguf"
TEXT_ENCODER_URL = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"
VAE_URL = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors"
DEFAULT_LORA_URL = "https://huggingface.co/conradlocke/krea2-identity-edit/resolve/main/krea2_identity_edit_v1_2.safetensors"

DIRS = {
    "unet":  os.path.join(WORKSPACE, "models/unet"),
    "clip":  os.path.join(WORKSPACE, "models/clip"),
    "text_encoders": os.path.join(WORKSPACE, "models/text_encoders"),
    "vae":   os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
    "input": os.path.join(WORKSPACE, "input"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir, filename=None):
    if not url.strip(): return
    if not filename:
        parsed_url = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed_url.path)
    if os.path.exists(os.path.join(target_dir, filename)) and os.path.getsize(os.path.join(target_dir, filename)) > 1000000:
        print(f"   ✅ Already Cached: {filename}")
        return

    print(f"   📥 Fetching: {filename}...")
    aria2_cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ]
    subprocess.run(aria2_cmd, check=True)
    print(f"   ✅ Saved in {os.path.basename(target_dir)}")

# Download required base assets
download_file(FIXED_UNET_URL, DIRS["unet"])
download_file(TEXT_ENCODER_URL, DIRS["clip"])
download_file(VAE_URL, DIRS["vae"])
download_file(DEFAULT_LORA_URL, DIRS["loras"])

# Symlink CLIP for Krea compatibility
clip_src = os.path.join(DIRS["clip"], "qwen3vl_4b_fp8_scaled.safetensors")
clip_dst = os.path.join(DIRS["text_encoders"], "qwen3vl_4b_fp8_scaled.safetensors")
if os.path.exists(clip_src) and not os.path.exists(clip_dst):
    os.symlink(clip_src, clip_dst)

# Handle optional extra LoRA
ACTIVE_LORA_FILENAME = "krea2_identity_edit_v1_2.safetensors"

if LORA_SOURCE == "Download from URL":
    if LORA_URLS.strip():
        parsed_name = os.path.basename(urllib.parse.urlparse(LORA_URLS.strip()).path)
        if not parsed_name.endswith(('.safetensors', '.pt', '.ckpt')):
            parsed_name = "downloaded_lora.safetensors"
        download_file(LORA_URLS.strip(), DIRS["loras"], filename=parsed_name)
        ACTIVE_LORA_FILENAME = parsed_name
    else:
        print("⚠️ No LoRA URL provided. Using default.")
elif LORA_SOURCE == "Upload from Computer":
    print("\n📤 [@CoinNoin] Select your extra LoRA (.safetensors) file to upload:")
    uploaded = files.upload()
    if uploaded:
        for fname in uploaded.keys():
            dest = os.path.join(DIRS["loras"], fname)
            os.replace(fname, dest)
            ACTIVE_LORA_FILENAME = fname
            print(f"   ✅ Extra LoRA uploaded: {fname}")
    else:
        print("⚠️ No file uploaded. Using default.")

available_loras = [f for f in os.listdir(DIRS["loras"]) if f.endswith(('.safetensors', '.pt', '.ckpt'))]
print("\n📂 [@CoinNoin] Available LoRAs in storage:")
for idx, lora_name in enumerate(available_loras, 1):
    print(f"   {idx}. {lora_name}")
    if idx == 1:
        recent_lora = lora_name

print("\n✅ [@CoinNoin] All core assets secured!")

⚡ [@CoinNoin] Configuring Aria2c Accelerator...
   ✅ Already Cached: krea2_turbo-Q6_K.gguf
   ✅ Already Cached: qwen3vl_4b_fp8_scaled.safetensors
   ✅ Already Cached: qwen_image_vae.safetensors
   ✅ Already Cached: krea2_identity_edit_v1_2.safetensors

📂 [@CoinNoin] Available LoRAs in storage:
   1. krea2_identity_edit_v1_2.safetensors

✅ [@CoinNoin] All core assets secured!


In [ ]:
#@title 3. Krea2Edit Image Generation
#@markdown Provide your instructions, upload your source image, and adjust editing sliders.

# --- Upload Controls ---
UPLOAD_SOURCE_IMAGE = True #@param {type:"boolean"}
#@markdown *If checked, Colab will prompt you to upload an image when you run this cell. Uncheck to use previous.*
UPLOAD_SECOND_REFERENCE = False #@param {type:"boolean"}
#@markdown *Optional: Upload a second reference image to guide the edit.*

# --- Generation Settings ---
EDIT_INSTRUCTION = "Change her outfit to a red raincoat." #@param {type:"string"}
WIDTH = 1024 #@param {type:"slider", min:512, max:1536, step:64}
HEIGHT = 1024 #@param {type:"slider", min:512, max:1536, step:64}

# --- Krea2Edit Advanced Settings ---
REF_BOOST = 4.0 #@param {type:"slider", min:0.5, max:10.0, step:0.1}
GROUNDING_PX = 768 #@param {type:"slider", min:384, max:1536, step:64}
FIT_MODE = "fit" #@param ["fit", "crop"]

# --- LoRA & Model Selection ---
#@markdown *Guide: Choose **auto** to use the most recent LoRA, or type the exact filename.*
LORA_FILENAME = "auto" #@param ["None", "auto"] {allow-input: true}
LORA_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}

# --- Advanced Sampler Settings ---
STEPS = 10 #@param {type:"slider", min:1, max:50, step:1}
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "euler" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_sde", "dpmpp_2m_sde", "dpmpp_3m_sde", "uni_pc", "lcm"] {allow-input: true}
SCHEDULER = "beta" #@param ["normal", "karras", "exponential", "sgm_uniform", "simple", "ddim_uniform", "beta"] {allow-input: true}
SEED = 0 #@param {type:"integer"}

# --- Direct Download (No ZIP) ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from PIL import Image
from IPython.display import display, Image as IPImage, clear_output

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
os.chdir(WORKSPACE)

# Fixed GGUF Model Filename for Krea-2
UNET_FILENAME = "krea2_turbo-Q6_K.gguf"

# Auto-detect LoRA if set to 'auto'
if LORA_FILENAME.lower() == "auto":
    if 'ACTIVE_LORA_FILENAME' in globals() and ACTIVE_LORA_FILENAME:
      LORA_FILENAME = ACTIVE_LORA_FILENAME
    else:
        LORA_FILENAME = "krea2_identity_edit_v1_2.safetensors"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py", "--port", "8188"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

# Handle Uploads
source_filename = "source_input.png"
second_filename = "second_input.png"
has_second_image = False

if UPLOAD_SOURCE_IMAGE:
    print("📤 [@CoinNoin] Please upload your SOURCE image...")
    uploaded = files.upload()
    if uploaded:
        source_name = list(uploaded.keys())[0]
        os.replace(source_name, os.path.join(INPUT_DIR, source_filename))
        print("   ✅ Source image saved!")
        saved_path = os.path.join(INPUT_DIR, source_filename)
        img = Image.open(saved_path)
        display(img.resize((300, int(300 * img.height / img.width))))

    else:
        print("   ❌ No image uploaded. Will attempt to use cached image.")
else:
    print("   ℹ️ Using previously uploaded source image.")

if UPLOAD_SECOND_REFERENCE:
    print("\n📤 [@CoinNoin] Please upload your SECOND REFERENCE image...")
    uploaded_sec = files.upload()
    if uploaded_sec:
        sec_name = list(uploaded_sec.keys())[0]
        os.replace(sec_name, os.path.join(INPUT_DIR, second_filename))
        has_second_image = True
        print("   ✅ Second image saved!")
elif os.path.exists(os.path.join(INPUT_DIR, second_filename)):
    has_second_image = True

# Brand visibility prefix in filename
BRAND_PREFIX = "@CoinNoin_KreaEdit"

print(f"\n\033[94m➜ [@CoinNoin] Details | Res: {WIDTH}x{HEIGHT} | Seed: {SEED} | Steps: {STEPS}\033[0m")
print(f"📝 Instruction: '{EDIT_INSTRUCTION}'")

# Build Krea2Edit Architecture
prompt_workflow = {
    "55": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET_FILENAME}},
    "56": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"}},
    "57": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
    "72": {"class_type": "LoadImage", "inputs": {"image": source_filename}},
    "73": {"class_type": "VAEEncode", "inputs": {"pixels": ["72", 0], "vae": ["57", 0]}},
    "82": {"class_type": "EmptySD3LatentImage", "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
}

model_output = ["55", 0]
if LORA_STRENGTH > 0 and LORA_FILENAME.lower() != "none":
    prompt_workflow["71"] = {
        "class_type": "LoraLoaderModelOnly",
        "inputs": {"lora_name": LORA_FILENAME, "strength_model": LORA_STRENGTH, "model": ["55", 0]}
    }
    model_output = ["71", 0]
    print(f"✨ [@CoinNoin] LoRA Applied: {LORA_FILENAME} (Strength: {LORA_STRENGTH})")

patch_inputs = {
    "model": model_output,
    "source_latent": ["73", 0],
    "vae": ["57", 0],
    "source_image": ["72", 0],
    "target_latent": ["82", 0],
    "ref_boost": REF_BOOST,
    "ref_boost_a": 1.0,
    "fit_mode": FIT_MODE
}

encode_inputs_pos = {"clip": ["56", 0], "prompt": EDIT_INSTRUCTION, "image": ["72", 0], "grounding_px": GROUNDING_PX}
encode_inputs_neg = {"clip": ["56", 0], "prompt": "", "image": ["72", 0], "grounding_px": GROUNDING_PX}

if has_second_image and os.path.exists(os.path.join(INPUT_DIR, second_filename)):
    prompt_workflow["90"] = {"class_type": "LoadImage", "inputs": {"image": second_filename}}
    prompt_workflow["92"] = {"class_type": "VAEEncode", "inputs": {"pixels": ["90", 0], "vae": ["57", 0]}}
    patch_inputs["source_latent_b"] = ["92", 0]
    patch_inputs["source_image_b"] = ["90", 0]
    encode_inputs_pos["image_b"] = ["90", 0]
    encode_inputs_neg["image_b"] = ["90", 0]
    print("✨ [@CoinNoin] Second Reference Image Attached!")

prompt_workflow["79"] = {"class_type": "Krea2EditModelPatch", "inputs": patch_inputs}
prompt_workflow["84"] = {"class_type": "Krea2EditGroundedEncode", "inputs": encode_inputs_pos}
prompt_workflow["85"] = {"class_type": "Krea2EditGroundedEncode", "inputs": encode_inputs_neg}

prompt_workflow["53"] = {
    "class_type": "KSampler",
    "inputs": {
        "seed": SEED, "steps": STEPS, "cfg": CFG, "sampler_name": SAMPLER_NAME,
        "scheduler": SCHEDULER, "denoise": 1.0, "model": ["79", 0],
        "positive": ["84", 0], "negative": ["85", 0], "latent_image": ["82", 0]
    }
}

prompt_workflow["54"] = {"class_type": "VAEDecode", "inputs": {"samples": ["53", 0], "vae": ["57", 0]}}
prompt_workflow["29"] = {"class_type": "SaveImage", "inputs": {"filename_prefix": BRAND_PREFIX, "images": ["54", 0]}}

p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("🖼️ Decoding Final Masterpiece...")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

# Direct download trigger
if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)